# Reproducible RTHYM-MOC Runs

This notebook demonstrates two related ideas:

1. Running the same RTHYM-MOC model twice with identical inputs gives identical outputs.
2. When synthetic inputs are useful, a fixed NumPy seed makes those generated scenarios reproducible too.

The notebook uses checked-in benchmark assets from this repository so the workflow stays tied to the same deterministic inputs used by the automated regression suite.

## 1. Install and Import Dependencies

This repo already contains the package source, benchmark assets, and examples. The next cell resolves the repository root, imports the Python libraries used in the notebook, and reports the package version that produced the results.

In [ ]:
from pathlib import Path
import json

import numpy as np

import rthym_moc
from rthym_moc import load_inp

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "tests").exists():
    REPO_ROOT = Path("/home/jason/RTHYM-MOC")

EXAMPLES_DIR = REPO_ROOT / "examples"
TESTS_DIR = REPO_ROOT / "tests"
OUTPUT_DIR = EXAMPLES_DIR / "notebook_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

ASSET_INP = TESTS_DIR / "Joukowsky Benchmark.inp"
ASSET_JSON = TESTS_DIR / "R-THYM_Joukowsky_Verification.json"

print(f"Repository root: {REPO_ROOT}")
print(f"Using INP asset : {ASSET_INP.name}")
print(f"Using JSON asset: {ASSET_JSON.name}")
print(f"rthym_moc version: {getattr(rthym_moc, '__version__', 'unknown')}")

## 2. Load a Deterministic Network Model

The benchmark suite already ships a fixed EPANET input file and a checked-in JSON export from the R-THYM reference run. The next cell uses those committed assets to build a solver with deterministic initial heads, flows, and valve schedule values.

In [ ]:
reference = json.loads(ASSET_JSON.read_text())
steady_nodes = reference["steadyState"]["nodes"]
steady_pipes = reference["steadyState"]["pipes"]
valve_schedule = [
    (float(point["t"]), float(point["pct"]))
    for point in reference["valveSchedules"]["Valve_A"]
]

BASE_INITIAL_FLOWS = {
    "Pipe_1": float(steady_pipes["Pipe_1"]["Q_gpm"]),
    "Pipe_2": float(steady_pipes.get("Pipe_2", steady_pipes["Pipe_1"])["Q_gpm"]),
    "Valve_A": float(steady_pipes["Pipe_1"]["Q_gpm"]),
}
BASE_INITIAL_HEADS = {
    "PressureBoundary_A": float(steady_nodes["PressureBoundary_A"]["head"]),
    "PressureBoundary_B": float(steady_nodes["PressureBoundary_B"]["head"]),
    "_VALVE_Valve_A": float(steady_nodes["Valve_A"]["head"]),
}

SIM_TOTAL_TIME = 8.0
SIM_DT = 0.01


def make_baseline_solver() -> rthym_moc.MOCSolver:
    solver = load_inp(
        str(ASSET_INP),
        use_wntr=False,
        initial_flows=BASE_INITIAL_FLOWS,
        initial_heads=BASE_INITIAL_HEADS,
    )
    solver.set_valve_schedule("_VALVE_Valve_A", valve_schedule)
    return solver


baseline_solver = make_baseline_solver()
baseline_results = baseline_solver.run(total_time=SIM_TOTAL_TIME, dt=SIM_DT)
node_ids = sorted(baseline_results["node_head"].keys())
pipe_ids = sorted(baseline_results["pipe_flow_gpm"].keys())

print("Node IDs:", node_ids)
print("Pipe IDs:", pipe_ids)
print("Valve schedule points:", len(valve_schedule))
print("Time steps in baseline run:", len(baseline_results["time"]))

## 3. Run the Solver and Capture Baseline Outputs

This first run captures the time series we care about for a deterministic benchmark comparison: the inline valve head and pressure, plus the upstream pipe flow.

In [ ]:
baseline_time = np.asarray(baseline_results["time"])
baseline_head = np.asarray(baseline_results["node_head"]["_VALVE_Valve_A"])
baseline_pressure = np.asarray(baseline_results["node_pressure"]["_VALVE_Valve_A"])
baseline_flow = np.asarray(baseline_results["pipe_flow_gpm"]["Pipe_1"])

pipe_diameter_ft = 12.0 / 12.0
pipe_area_ft2 = np.pi * (pipe_diameter_ft / 2.0) ** 2
initial_velocity = float(BASE_INITIAL_FLOWS["Pipe_1"] * rthym_moc.GPM_TO_CFS / pipe_area_ft2)
approx_wave_speed = 4000.0
expected_head_rise = approx_wave_speed * initial_velocity / rthym_moc.G_FT_S2
expected_peak_head = BASE_INITIAL_HEADS["PressureBoundary_A"] + expected_head_rise

baseline_frame = {
    "time_s": baseline_time,
    "valve_head_ft": baseline_head,
    "valve_pressure_psi": baseline_pressure,
    "pipe_1_flow_gpm": baseline_flow,
}
print("First five head values:", baseline_head[:5])

print(f"Expected Joukowsky peak head (approx): {expected_peak_head:.3f} ft")
print(f"Simulated peak head: {baseline_head.max():.3f} ft")

## 4. Repeat the Same Simulation to Verify Determinism

A fresh solver instance is built from the same committed inputs, then run again with identical settings. Rebuilding the solver avoids any ambiguity around in-memory state from the first run.

In [ ]:
repeat_solver = make_baseline_solver()
repeat_results = repeat_solver.run(total_time=SIM_TOTAL_TIME, dt=SIM_DT)

repeat_time = np.asarray(repeat_results["time"])
repeat_head = np.asarray(repeat_results["node_head"]["_VALVE_Valve_A"])
repeat_pressure = np.asarray(repeat_results["node_pressure"]["_VALVE_Valve_A"])
repeat_flow = np.asarray(repeat_results["pipe_flow_gpm"]["Pipe_1"])

print(f"Repeated run peak head: {repeat_head.max():.3f} ft")
print(f"Repeated run peak pressure: {repeat_pressure.max():.3f} psi")

## 5. Compare Time-Series Outputs Numerically

The next cell compares the repeated runs using exact equality and explicit max-difference checks. For this fixed-input case, the expected tolerance is zero because no random inputs are involved.

In [ ]:
determinism_metrics = {
    "max_abs_diff_time": float(np.max(np.abs(baseline_time - repeat_time))),
    "max_abs_diff_head_ft": float(np.max(np.abs(baseline_head - repeat_head))),
    "max_abs_diff_pressure_psi": float(np.max(np.abs(baseline_pressure - repeat_pressure))),
    "max_abs_diff_flow_gpm": float(np.max(np.abs(baseline_flow - repeat_flow))),
}

print(json.dumps(determinism_metrics, indent=2))
assert np.array_equal(baseline_time, repeat_time)
assert np.array_equal(baseline_head, repeat_head)
assert np.array_equal(baseline_pressure, repeat_pressure)
assert np.array_equal(baseline_flow, repeat_flow)
print("Repeated deterministic runs matched exactly.")

## 6. Optional: Seed NumPy for Synthetic Input Generation

The production solver and current tests do not rely on random inputs, but seeded synthetic data is still useful for benchmark matrices, perturbation studies, or fuzz-like scenario generation. The next cells keep the randomness explicit and reproducible with `numpy.random.default_rng`.

## 7. Generate Seeded Synthetic Demand Variations

This Joukowsky benchmark has no demand nodes, so the synthetic perturbation is applied to the downstream fixed-head boundary instead. The helper returns a reproducible head schedule for `PressureBoundary_B` from a chosen seed.

In [ ]:
SEEDED_TOLERANCE = 0.0
SCHEDULE_TIMES = np.array([0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
BASE_DOWNSTREAM_HEAD = BASE_INITIAL_HEADS["PressureBoundary_B"]


def make_seeded_head_schedule(seed: int, amplitude_ft: float = 0.35):
    rng = np.random.default_rng(seed)
    offsets = rng.normal(loc=0.0, scale=amplitude_ft, size=len(SCHEDULE_TIMES) - 1)
    schedule_values = np.concatenate(([BASE_DOWNSTREAM_HEAD], BASE_DOWNSTREAM_HEAD + offsets))
    schedule = list(zip(SCHEDULE_TIMES.tolist(), schedule_values.tolist()))
    return schedule, schedule_values


schedule_a, values_a = make_seeded_head_schedule(2026)
schedule_b, values_b = make_seeded_head_schedule(2026)
schedule_c, values_c = make_seeded_head_schedule(7)

print("Seed 2026 schedule preview:", schedule_a[:3])
assert np.array_equal(values_a, values_b)
assert not np.array_equal(values_a, values_c)
print("The same seed reproduced identical synthetic inputs; a different seed changed them.")

## 8. Re-run Simulations with Seeded Inputs

The same seeded schedule should reproduce identical outputs, while a different seed should change the generated boundary schedule and therefore the resulting traces.

In [ ]:
def run_seeded_case(seed: int):
    solver = make_baseline_solver()
    schedule, schedule_values = make_seeded_head_schedule(seed)
    solver.set_head_schedule("PressureBoundary_B", schedule)
    results = solver.run(total_time=SIM_TOTAL_TIME, dt=SIM_DT)
    return schedule_values, results


seeded_values_1, seeded_results_1 = run_seeded_case(2026)
seeded_values_2, seeded_results_2 = run_seeded_case(2026)
seeded_values_3, seeded_results_3 = run_seeded_case(7)

seeded_head_1 = np.asarray(seeded_results_1["node_head"]["_VALVE_Valve_A"])
seeded_head_2 = np.asarray(seeded_results_2["node_head"]["_VALVE_Valve_A"])
seeded_head_3 = np.asarray(seeded_results_3["node_head"]["_VALVE_Valve_A"])

same_seed_max_diff = float(np.max(np.abs(seeded_head_1 - seeded_head_2)))
different_seed_max_diff = float(np.max(np.abs(seeded_head_1 - seeded_head_3)))

print(f"Same-seed max head difference: {same_seed_max_diff:.6f} ft")
print(f"Different-seed max head difference: {different_seed_max_diff:.6f} ft")
assert np.array_equal(seeded_values_1, seeded_values_2)
assert np.array_equal(seeded_head_1, seeded_head_2)
assert same_seed_max_diff <= SEEDED_TOLERANCE
assert different_seed_max_diff > 0.0
print("Seeded synthetic runs are reproducible when the seed is fixed.")

## 9. Persist Reproducible Results for Regression Checks

The final cell writes selected outputs and metadata to disk so the notebook can double as a repeatable validation workflow and a starting point for future regression assets.

In [ ]:
metadata = {
    "package_version": getattr(rthym_moc, "__version__", "unknown"),
    "asset_inp": str(ASSET_INP.relative_to(REPO_ROOT)),
    "asset_json": str(ASSET_JSON.relative_to(REPO_ROOT)),
    "simulation_total_time_s": SIM_TOTAL_TIME,
    "simulation_dt_s": SIM_DT,
    "determinism_metrics": determinism_metrics,
    "seeded_tolerance_ft": SEEDED_TOLERANCE,
    "seeded_example_seed": 2026,
}

metadata_path = OUTPUT_DIR / "quickstart_notebook_metadata.json"
trace_path = OUTPUT_DIR / "quickstart_notebook_baseline_trace.csv"

metadata_path.write_text(json.dumps(metadata, indent=2))

trace_matrix = np.column_stack((baseline_time, baseline_head, baseline_pressure, baseline_flow))
np.savetxt(
    trace_path,
    trace_matrix,
    delimiter=",",
    header="time_s,valve_head_ft,valve_pressure_psi,pipe_1_flow_gpm",
    comments="",
)

print(f"Wrote metadata to {metadata_path}")
print(f"Wrote baseline trace to {trace_path}")